# الفصل 59: إدارة التكلفة ونضج LLMOps

> تكلفة تشغيل LLM في الإنتاج يمكن أن تتصاعد بسرعة مذهلة دون رقابة.

- **الجزء:** 11 — النشر والتشغيل
- **الفصل:** 59
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 58: الأمان والحوكمة في الإنتاج
- **الوسوم:** Cost Engineering، LLMOps، ROI، Caching

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch59-cost-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-11-llmops%2Fch59-cost-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-11-llmops/ch59-cost-optimization.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> تكلفة تشغيل LLM في الإنتاج يمكن أن تتصاعد بسرعة مذهلة دون رقابة.

> **بيانات الفصل**
>
> - **الجزء:** 11 — النشر والتشغيل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 58: الأمان والحوكمة في الإنتاج


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم نمذجة التكلفة الشاملة
- فهم استراتيجيات تقليل التكلفة العملية
- فهم Cost Engineering


## 1. لماذا يهمّك هذا الموضوع؟

تكلفة تشغيل LLM في الإنتاج يمكن أن تتصاعد بسرعة مذهلة دون رقابة. هذا الفصل الختامي للجزء يُعلّمك كيف تُدير هذه التكلفة بذكاء، وكيف تبني نضجًا تشغيليًا كاملاً (LLMOps) يضمن استدامة النظام طويل المدى.


## 2. الخريطة الذهنية


```text
اقتصاديات LLM
├── مكوّنات التكلفة
│   ├── GPU/Compute
│   ├── التخزين والشبكة
│   ├── الاستدعاءات الخارجية
│   ├── البشر (تشغيل/مراجعة)
│   └── الأدوات والمراقبة
├── اقتصاد الوحدة
│   ├── تكلفة/طلب
│   ├── تكلفة/مهمة ناجحة
│   └── تكلفة/مستخدم شهريًا
├── الرافعات
│   ├── نموذج أصغر/توجيه
│   ├── تكميم
│   ├── تخزين مؤقت
│   ├── تقليل tokens
│   └── دفعات/Spot
└── الحوكمة
    ├── ميزانيات وحدود
    ├── Chargeback
    └── مراجعة شهرية
```


## 3. الشرح المتدرج

### 🟢 أ) نمذجة التكلفة الشاملة


In [ ]:
class LLMCostCalculator:
    """حاسبة تكلفة شاملة لمقارنة خيارات النشر المختلفة"""

    def calculate_self_hosted_cost(self, gpu_type: str, num_gpus: int,
                                      hours_per_month: int, requests_per_month: int) -> dict:
        gpu_hourly_rates = {"A100-80GB": 2.5, "H100": 4.5, "A10G": 1.0}   # تقديرية، تختلف حسب المزوّد

        compute_cost = gpu_hourly_rates[gpu_type] * num_gpus * hours_per_month
        cost_per_request = compute_cost / requests_per_month if requests_per_month > 0 else 0

        return {
            "monthly_compute_cost": compute_cost,
            "cost_per_request": cost_per_request,
            "break_even_analysis": "مربح مقارنة بـ API إذا كان الحجم مرتفعًا بثبات"
        }

    def calculate_api_cost(self, provider_rate_per_1k_input: float,
                              provider_rate_per_1k_output: float,
                              avg_input_tokens: int, avg_output_tokens: int,
                              requests_per_month: int) -> dict:
        input_cost = (avg_input_tokens / 1000) * provider_rate_per_1k_input * requests_per_month
        output_cost = (avg_output_tokens / 1000) * provider_rate_per_1k_output * requests_per_month
        total = input_cost + output_cost

        return {
            "monthly_api_cost": total,
            "cost_per_request": total / requests_per_month if requests_per_month > 0 else 0
        }

    def recommend_deployment_strategy(self, requests_per_month: int,
                                         avg_tokens_per_request: int) -> str:
        """دليل قرار مبسّط: متى الاستضافة الذاتية أرخص من استخدام API؟"""
        self_hosted = self.calculate_self_hosted_cost("A100-80GB", 2, 730, requests_per_month)
        api_based = self.calculate_api_cost(0.01, 0.03, avg_tokens_per_request,
                                              avg_tokens_per_request, requests_per_month)

        if self_hosted["monthly_compute_cost"] < api_based["monthly_api_cost"]:
            return f"✅ الاستضافة الذاتية أرخص: ${self_hosted['monthly_compute_cost']:.0f} مقابل ${api_based['monthly_api_cost']:.0f}"
        return f"✅ استخدام API أرخص عند هذا الحجم: ${api_based['monthly_api_cost']:.0f} مقابل ${self_hosted['monthly_compute_cost']:.0f}"


### 🟡 ب) استراتيجيات تقليل التكلفة العملية


In [ ]:
class ResponseCache:
    """
    التخزين المؤقت للاستجابات — أبسط وأفعل تقنية لتقليل التكلفة
    خاصة للأسئلة المتكررة (FAQ-style) أو نفس السياق مع System Prompt ثابت
    """
    def __init__(self, embedder, similarity_threshold=0.95, max_cache_size=10000):
        self.embedder = embedder
        self.threshold = similarity_threshold
        self.cache = []   # [{"prompt_embedding": ..., "prompt": ..., "response": ...}]
        self.max_size = max_cache_size
        self.hit_count, self.miss_count = 0, 0

    def get_cached_response(self, prompt: str) -> str:
        """بحث دلالي (Semantic Caching) — وليس تطابقًا حرفيًا فقط"""
        if not self.cache:
            self.miss_count += 1
            return None

        query_embedding = self.embedder.encode([prompt])[0]
        similarities = [
            np.dot(query_embedding, entry["prompt_embedding"]) for entry in self.cache
        ]
        max_sim_idx = np.argmax(similarities)

        if similarities[max_sim_idx] >= self.threshold:
            self.hit_count += 1
            return self.cache[max_sim_idx]["response"]

        self.miss_count += 1
        return None

    def store(self, prompt: str, response: str):
        if len(self.cache) >= self.max_size:
            self.cache.pop(0)   # إزالة الأقدم (يمكن استبدالها بـ LRU حقيقي)
        self.cache.append({
            "prompt_embedding": self.embedder.encode([prompt])[0],
            "prompt": prompt, "response": response
        })

    def get_hit_rate(self) -> float:
        total = self.hit_count + self.miss_count
        return self.hit_count / total if total > 0 else 0

class ModelRouter:
    """
    توجيه ذكي بين نماذج مختلفة الحجم/التكلفة حسب تعقيد السؤال
    (نموذج صغير رخيص للأسئلة البسيطة، نموذج كبير فقط للمعقّدة)
    """
    def __init__(self, small_model, large_model, complexity_classifier):
        self.small_model = small_model
        self.large_model = large_model
        self.complexity_classifier = complexity_classifier

    def route_and_generate(self, prompt: str) -> dict:
        complexity_score = self.complexity_classifier(prompt)   # 0 (بسيط) - 1 (معقّد جدًا)

        if complexity_score < 0.3:
            response = self.small_model.generate(prompt)
            model_used = "small (رخيص)"
        else:
            response = self.large_model.generate(prompt)
            model_used = "large (مكلف لكن أدق)"

        return {"response": response, "model_used": model_used, "complexity_score": complexity_score}


> 💡 **إحصائية صناعية مهمة**: دراسات حالة من شركات إنتاجية كبرى تُظهر أن **Semantic Caching** يمكن أن يُقلّل تكلفة الاستدلال بنسبة 20-40% في تطبيقات بها تكرار طبيعي في الأسئلة (دعم العملاء، الأسئلة الشائعة)، بينما **Model Routing** الذكي (استخدام نموذج صغير رخيص لـ 70-80% من الاستعلامات البسيطة) يحقق توفيرًا مشابهًا أو أكبر دون التضحية بجودة معالجة الحالات المعقّدة.

### 🔴 ج) طبقة التعمّق: نضج LLMOps — دورة الحياة الكاملة وCI/CD للنماذج


In [ ]:
"""
خط CI/CD كامل لنموذج LLM — يدمج كل ما تعلمناه في هذا الجزء والجزء التاسع
"""

LLMOPS_PIPELINE_YAML = """
name: llm-deployment-pipeline

stages:
  - stage: data_validation
    steps:
      - run: python validate_training_data.py --check-contamination --check-pii

  - stage: training_or_finetuning
    steps:
      - run: python train.py --config configs/production.yaml
      - run: python save_checkpoint.py --to-model-registry

  - stage: offline_evaluation
    steps:
      - run: python run_evaluation_suite.py --benchmarks mmlu,gsm8k,arabic_mmlu
      - run: python run_safety_redteam.py
      - run: python run_llm_judge_comparison.py --baseline=current_prod_model
      - gate: "REJECTED_SMOKE_TEST_FAILURE أو REGRESSION_DETECTED يوقف الخط فورًا"

  - stage: staging_deployment
    steps:
      - run: deploy_to_staging.sh
      - run: python run_integration_tests.py

  - stage: canary_deployment
    steps:
      - run: deploy_canary.sh --traffic-percentage=5
      - monitor: "لمدة 24 ساعة: latency, error_rate, quality_scores"
      - gate: "أي انحراف إحصائي سلبي يُوقف الزيادة ويُفعّل rollback تلقائي"

  - stage: progressive_rollout
    steps:
      - run: increase_traffic.sh --to=25
      - run: increase_traffic.sh --to=50
      - run: increase_traffic.sh --to=100

  - stage: post_deployment_monitoring
    steps:
      - run: enable_continuous_quality_monitoring.py
      - run: enable_drift_detection.py
"""

class LLMOpsMaturityAssessment:
    """
    إطار تقييم ذاتي لنضج ممارسات LLMOps في المنظمة
    """
    MATURITY_LEVELS = {
        1: "يدوي بالكامل — نشر يدوي، لا مراقبة منهجية، لا نسخ احتياطي للنماذج",
        2: "آلي جزئيًا — CI/CD أساسي، مراقبة تقنية فقط (latency/errors)",
        3: "آلي شامل — تقييم آلي قبل النشر، canary deployment، مراقبة جودة أساسية",
        4: "استباقي — كشف انجراف مستمر، A/B testing منهجي، تحسين تكلفة آلي",
        5: "ذاتي التحسّن — حلقة تغذية راجعة من الإنتاج تُغذّي تحسينات تدريب مستقبلية تلقائيًا"
    }

    def assess(self, organization_practices: dict) -> dict:
        score = 1
        if organization_practices.get("has_cicd"): score = max(score, 2)
        if organization_practices.get("has_automated_evaluation"): score = max(score, 3)
        if organization_practices.get("has_canary_deployment"): score = max(score, 3)
        if organization_practices.get("has_drift_detection"): score = max(score, 4)
        if organization_practices.get("has_feedback_loop_to_training"): score = max(score, 5)

        return {"maturity_level": score, "description": self.MATURITY_LEVELS[score],
                "next_step": self.MATURITY_LEVELS.get(score + 1, "أنت في أعلى مستوى نضج")}


## 4. مثال محلول خطوة بخطوة

**مقارنة: استضافة ذاتية vs API.**

**الاستضافة الذاتية:** A100 بسعر 2 دولار/ساعة، throughput مقاس 2,000 token/s عند استخدام 100%، واستخدام فعلي 50%:

\[
\text{Tokens/day} = 2{,}000 \times 0.5 \times 86{,}400 = 86.4 \text{ مليون}
\]

\[
\text{Cost/1M} = \frac{48}{86.4} \approx 0.56 \text{ دولار}
\]

(48 دولارًا = 2 × 24 ساعة).

**API:** سعر مختلط 1 دولار لكل مليون token (مثال توضيحي).

| البند | استضافة ذاتية | API |
|---|---:|---:|
| تكلفة 1M token | 0.56 دولار | 1.00 دولار |
| تكلفة ثابتة | عالية (عتاد/فريق) | لا |
| مرونة الحمل | محدودة | عالية |
| تحكم/خصوصية | كامل | حسب المزود |
| وقت الإعداد | أسابيع | ساعات |

**نقطة التعادل:** إذا كانت التكلفة الثابتة الشهرية للاستضافة (عتاد + تشغيل) 5,000 دولار، فإنها تعادل 5,000 / (1.00 - 0.56) ≈ 11.4 مليون token شهريًا. فوق هذا الحجم، قد تصبح الاستضافة الذاتية أرخص؛ تحتها، الـ API غالبًا أفضل.

**الدرس:** القرار ليس «أرخص سعر لكل token»، بل «أقل تكلفة إجمالية مع مستوى الجودة والامتثال المطلوب».


## 5. نصائح وحيل

> 💡 **قِس تكلفة المهمة الناجحة لا تكلفة الطلب**: طلب فاشل أُعيد 3 مرات أغلى من طلب ناجح واحد. أضف معدل النجاح إلى المعادلة.

> 💡 **وجّه الطلبات حسب التعقيد**: نموذج صغير للأسئلة البسيطة، ونموذج كبير للحالات الحرجة. التوجيه الذكي يوفّر 40-70% غالبًا.

> 💡 **قلّل الـ prompt قبل التكميم**: حذف السياق غير الضروري وتقصير التعليمات يوفّر أكثر من أي تحسين عتاد، وبتكلفة هندسية أقل.

> 💡 **استخدم التخزين المؤقت بحذر**: cache للإجابات المتكررة وللبادئات. أضف TTL ومفتاحًا يشمل النسخة والمستخدم، وإلا فقد تُقدّم إجابة قديمة.

> 💡 **راقب التكلفة يوميًا مع ميزانية**: حدّد سقفًا شهريًا وتنبيهًا عند 50/80/100%. التكلفة تنفلت بصمت في أنظمة الوكلاء.


## 6. أخطاء شائعة ومصائد

> ⚠️ **تحسين GPU فقط**: قد تكون تكاليف الهندسة البشرية والمراجعة أكبر من تكلفة الحوسبة. احسب التكلفة الكلية.

> ⚠️ **إغفال الخمول**: نسخ GPU تعمل بلا طلبات. استخدم autoscaling إلى صفر للبيئات غير الإنتاجية، أو جدولة خارج ساعات العمل.

> ⚠️ **التخزين المؤقت الأعمى**: إجابة مخزّنة لسياسة تغيّرت منذ شهر = خطر. اربط الـ cache بإصدار المعرفة.

> ⚠️ **الادعاء بأن التكميم «مجاني»**: قد يقلّل الجودة في مهام دقيقة. قِس الجودة قبل وبعد، ولا تعتمد على أرقام المزود.

> ⚠️ **غياب Chargeback**: بدون نسب التكلفة للفرق/الميزات، لا أحد مسؤول. أضف وسومًا لكل طلب وميزانية لكل فريق.


## 7. جدول مقارنة

| الاستراتيجية | التوفير المحتمل | التعقيد التنفيذي | المقايضة |
|---|---|---|---|
| **Semantic Caching** | 20-40% (حسب تكرار الأسئلة) | منخفض | يتطلب ضبط دقيق لعتبة التشابه |
| **Model Routing** | 30-50% | متوسط | يتطلب مصنّف تعقيد دقيق |
| **Quantization** | 50-75% (ذاكرة/حوسبة) | منخفض-متوسط | فقدان جودة طفيف محتمل |
| **Speculative Decoding** | 50-70% (زمن فقط، ليس تكلفة GPU مباشرة) | عالٍ | يتطلب نموذج draft متوافق |
| **Batch Processing (غير تفاعلي)** | 40-60% | منخفض | غير مناسب للتطبيقات التفاعلية الفورية |
| **اختيار حجم نموذج أصغر كافٍ** | متغيّر جدًا (قد يكون الأكبر) | منخفض | يتطلب تقييمًا دقيقًا لعدم كفاية الأداء |


## 8. ملخص أفضل الممارسات

- [ ] قياس التكلفة الفعلية لكل طلب (cost-per-request) بشكل مستمر، وليس تقديرًا نظريًا فقط
- [ ] تطبيق Semantic Caching على الأقل لحالات الاستخدام ذات التكرار الطبيعي
- [ ] تقييم Model Routing إذا كان توزيع تعقيد الأسئلة غير متجانس
- [ ] خط CI/CD كامل يدمج التقييم الآلي (الجزء التاسع) مع النشر التدريجي
- [ ] تقييم دوري لمستوى نضج LLMOps في المنظمة، وخطة واضحة للانتقال للمستوى التالي

---


## 9. أسئلة مقابلات

**سؤال 1: كيف تحسب تكلفة طلب LLM؟**

> **إجابة نموذجية:** التكلفة = (tokens المدخلات × سعر الإدخال) + (tokens المخرجات × سعر الإخراج) + تكاليف البنية (GPU/شبكة/تخزين) موزعة على الطلبات + تكاليف المراجعة. في الاستضافة الذاتية، أحسب تكلفة GPU/ساعة وأقسمها على throughput الفعلي. أضيف معدل إعادة المحاولات والفشل. الأهم: تكلفة المهمة الناجحة، لا تكلفة الاستدعاء الواحد.

**سؤال 2: متى تختار الاستضافة الذاتية؟**

> **إجابة نموذجية:** عندما يكون الحجم كبيرًا بما يكفي لتجاوز نقطة التعادل، أو عندما تكون الخصوصية/الامتثال تتطلب عدم إرسال البيانات لمزود خارجي، أو عندما تحتاج تحكمًا كاملًا في النموذج والتخصيص. يجب أن يتوفر فريق تشغيل وتحتية ومراقبة. إذا كان الحجم صغيرًا أو الفريق محدودًا، فالـ API أسرع وأقل مخاطرة.

**سؤال 3: كيف تقلّل التكلفة دون خسارة الجودة؟**

> **إجابة نموذجية:** أبدأ بالقياس: أين تُصرف التكلفة؟ ثم: تقليل tokens (سياق أقصر، مخرجات محددة)، توجيه النماذج حسب الصعوبة، caching للطلبات المتكررة، تكميم مع قياس الجودة، واستخدام Spot/autoscaling. أختبر كل تغيير على مجموعة تقييم، وأقبل التوفير فقط إذا بقي ضمن حدود الجودة المتفق عليها.


## 10. تمرين عملي

**الهدف:** بناء نموذج تكلفة لخدمة مليون طلب شهريًا.

**الخطوات:**
1. افترض متوسط 500 token إدخال و300 token إخراج لكل طلب.
2. احسب التكلفة الشهرية لثلاثة خيارات: API كبير، API صغير، واستضافة ذاتية.
3. أضف تكاليف المراقبة والتخزين والمراجعة البشرية (تقديرية).
4. احسب تكلفة الطلب وتكلفة المهمة الناجحة (بافتراض 90% نجاح).
5. حدّد نقطة التعادل بين الخيارات.
6. اقترح خطة توفير من 3 خطوات مع أثر متوقع على التكلفة والجودة.

**معيار النجاح:** جدول مقارنة واضح، ونقطة تعادل محسوبة، وتوصية مبررة.


## 11. مراجع للتعمّق

- FinOps Foundation. *FinOps for AI*. finops.org.
- Kwon, W., et al. (2023). *vLLM: Easy, Fast, and Cheap LLM Serving with PagedAttention*. SOSP.
- Hugging Face. *LLM Inference Optimization* and *Cost Calculator*.
- Paleyes, A., Urma, R.-G., & Lawrence, N. D. (2022). *Challenges in Deploying Machine Learning: A Survey of Case Studies*. ACM Computing Surveys.
- Sculley, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النشر والتشغيل / الفصل 59: إدارة التكلفة ونضج LLMOps.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
